In [1]:
!pip install -q pulp streamlit pandas plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 23.3 MB/s eta 0:00:00


In [8]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pulp
import plotly.express as px

st.set_page_config(page_title="Corporate budget optimizer", layout="wide")

st.title("📈 Corporate capital budgeting optimizer")
st.markdown("""
Această aplicație utilizează **Integer Linear Programming (ILP)** pentru a selecta portofoliul optim de proiecte,
maximizând **ROI-ul (Return on Investment)** sub constrângeri stricte de buget și resurse.
""")

data = {
    'Project_ID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    'Department': ['IT', 'IT', 'IT', 'Marketing', 'Marketing', 'Marketing', 'R&D', 'R&D', 'R&D', 'HR', 'HR', 'Logistics'],
    'Project_Name': [
        'Cloud Migration', 'Cybersecurity Upgrade', 'New Laptops',
        'TV Ad Campaign', 'Social Media Boost', 'Rebranding',
        'AI Model Dev', 'Prototype Testing', 'Patent Filing',
        'Employee Training', 'Recruitment Soft', 'Warehouse Automation'
    ],
    'Cost_EUR': [12000, 8000, 5000, 15000, 3000, 10000, 20000, 7000, 4000, 2000, 5000, 18000],
    'Expected_ROI_EUR': [25000, 18000, 6000, 30000, 8000, 15000, 50000, 15000, 9000, 5000, 9000, 40000],
    'Strategic_Score': [9, 10, 3, 8, 6, 5, 10, 7, 8, 4, 6, 9]
}

df = pd.DataFrame(data)

st.sidebar.header("Parametri de optimizare")

budget = st.sidebar.number_input("Buget total disponibil (€)", min_value=10000, max_value=200000, value=60000, step=5000)

st.sidebar.subheader("Reguli de business")
min_projects_dept = st.sidebar.checkbox("Asigură diversificare (minim un proiect/departament)", value=True)
max_projects = st.sidebar.slider("Numarul maxim de proiecte totale", 5, 12, 10)

st.subheader("Portofoliu de proiecte propuse")
st.dataframe(df.style.format({"Cost_EUR": "€{:,}", "Expected_ROI_EUR": "€{:,}"}))

if st.button("Optimizeaza alocarea bugetului", type="primary"):

    # Initializare problema (maximizare)
    prob = pulp.LpProblem("Maximize_ROI", pulp.LpMaximize)

    # Variabile de decizie (1=alegem proiectul, 0=nu alegem proiectul)
    project_vars = pulp.LpVariable.dicts("Select", df.index, cat='Binary')

    # Functia Obiectiv: maximizăm ROI-ul total
    prob += pulp.lpSum([df['Expected_ROI_EUR'][i] * project_vars[i] for i in df.index])

    # Constrangeri:
    prob += pulp.lpSum([df['Cost_EUR'][i] * project_vars[i] for i in df.index]) <= budget

    prob += pulp.lpSum([project_vars[i] for i in df.index]) <= max_projects

    if min_projects_dept:
        departments = df['Department'].unique()
        for dept in departments:
            dept_indices = df[df['Department'] == dept].index
            prob += pulp.lpSum([project_vars[i] for i in dept_indices]) >= 1

    # Rezolvarea problemei
    prob.solve()
    status = pulp.LpStatus[prob.status]

    if status == "Optimal":
        st.success(f"Solutie optimă gasita!")

        selection = [i for i in df.index if project_vars[i].varValue == 1]
        selected_df = df.iloc[selection].copy()

        total_cost = selected_df['Cost_EUR'].sum()
        total_roi = selected_df['Expected_ROI_EUR'].sum()
        roi_percent = ((total_roi - total_cost) / total_cost) * 100

        col1, col2, col3 = st.columns(3)
        col1.metric("Buget utilizat", f"€{total_cost:,}", f"din €{budget:,}")
        col2.metric("ROI total generat", f"€{total_roi:,}")
        col3.metric("Randament (yield)", f"{roi_percent:.1f}%")

        col_chart1, col_chart2 = st.columns(2)

        with col_chart1:
            st.subheader("Alocare buget per departament")
            fig_pie = px.pie(selected_df, names='Department', values='Cost_EUR', hole=0.4)
            st.plotly_chart(fig_pie, use_container_width=True)

        with col_chart2:
            st.subheader("Proiecte selectate vs Proiecte respinse")
            df['Status'] = ['Aprobat' if i in selection else 'Respins' for i in df.index]
            fig_bar = px.bar(df, x='Project_Name', y='Expected_ROI_EUR', color='Status',
                             color_discrete_map={'Aprobat': 'green', 'Respins': 'red'})
            st.plotly_chart(fig_bar, use_container_width=True)

        st.subheader("Lista finala de proiecte aprobate")
        st.dataframe(selected_df[['Department', 'Project_Name', 'Cost_EUR', 'Expected_ROI_EUR']].style.format({"Cost_EUR": "€{:,}", "Expected_ROI_EUR": "€{:,}"}))

    else:
        st.error("Nu s-a gasit o solutie optima. Incearca sa maresti bugetul.")

Overwriting app.py


In [10]:
import time
import subprocess

process = subprocess.Popen(["streamlit", "run", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &

time.sleep(8)

!grep -o 'https://.*\.trycloudflare.com' cloudflared.log | head -n 1

while True:
    time.sleep(1)

https://vids-temperatures-screens-cake.trycloudflare.com


KeyboardInterrupt: 